In [41]:
!pip install xgboost --quiet


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [42]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from xgboost import XGBClassifier



In [43]:
df = pd.read_csv('developer_burnout_dataset_7000.csv')

In [44]:
df.head()

,age,experience_years,daily_work_hours,sleep_hours,caffeine_intake,bugs_per_day,commits_per_day,meetings_per_day,screen_time,exercise_hours,stress_level,burnout_level
0,26.0,12.0,10.33,4.45,2.0,11.0,4.0,1.0,15.07,0.14,55.96,Medium
1,39.0,10.0,8.62,5.77,5.0,15.0,11.0,5.0,13.25,0.54,82.22,High
2,34.0,13.0,NaN,4.03,5.0,2.0,18.0,9.0,11.18,1.54,61.77,Medium
3,30.0,1.0,6.85,6.47,2.0,15.0,26.0,1.0,11.14,0.96,54.98,Medium
4,27.0,7.0,4.24,5.80,NaN,9.0,17.0,7.0,8.05,0.36,27.90,Low


In [45]:
df.columns

Index(['age', 'experience_years', 'daily_work_hours', 'sleep_hours',
       'caffeine_intake', 'bugs_per_day', 'commits_per_day',
       'meetings_per_day', 'screen_time', 'exercise_hours', 'stress_level',
       'burnout_level'],
      dtype='object')

In [46]:
# df[df.isnull().any(axis=1)]
df.isnull().sum()


age                 140
experience_years    140
daily_work_hours    140
sleep_hours         140
caffeine_intake     140
bugs_per_day        140
commits_per_day     140
meetings_per_day    140
screen_time         140
exercise_hours      140
stress_level        140
burnout_level       140
dtype: int64

In [47]:
df = df.fillna(df.median(numeric_only=True))

In [48]:
df.isnull().sum()

age                   0
experience_years      0
daily_work_hours      0
sleep_hours           0
caffeine_intake       0
bugs_per_day          0
commits_per_day       0
meetings_per_day      0
screen_time           0
exercise_hours        0
stress_level          0
burnout_level       140
dtype: int64

In [49]:
df = df.fillna(df.mode().iloc[0])

In [50]:
df.isnull().sum()

age                 0
experience_years    0
daily_work_hours    0
sleep_hours         0
caffeine_intake     0
bugs_per_day        0
commits_per_day     0
meetings_per_day    0
screen_time         0
exercise_hours      0
stress_level        0
burnout_level       0
dtype: int64

In [51]:
X = df.drop(["burnout_level", "stress_level"], axis=1)
y = df["burnout_level"]

In [52]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=3)

In [53]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Training on Logistic Regression

In [63]:
def run_logistic_regression_model(X_train, 
                                  y_train, 
                                  X_test, 
                                  y_test,
                                  max_iter=1000, 
                                  class_weight=None):
    
    model = LogisticRegression(max_iter=max_iter, class_weight=class_weight)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))
    
    return model

In [64]:
model = run_logistic_regression_model(X_train_scaled, y_train, X_test_scaled, y_test)

              precision    recall  f1-score   support

        High       0.80      0.70      0.75       535
         Low       0.82      0.73      0.77       478
      Medium       0.76      0.85      0.80      1087

    accuracy                           0.78      2100
   macro avg       0.79      0.76      0.77      2100
weighted avg       0.78      0.78      0.78      2100



Balanceando classe

In [65]:
model = run_logistic_regression_model(X_train_scaled, y_train, X_test_scaled, y_test, class_weight="balanced")

              precision    recall  f1-score   support

        High       0.72      0.82      0.77       535
         Low       0.71      0.84      0.77       478
      Medium       0.81      0.70      0.75      1087

    accuracy                           0.76      2100
   macro avg       0.75      0.78      0.76      2100
weighted avg       0.77      0.76      0.76      2100



In [91]:
data = {
    'age': 40.0,
    'experience_years': 20.0,
    'daily_work_hours': 8.0,
    'sleep_hours': 9.0,
    'caffeine_intake': 1.0,
    'bugs_per_day': 10.0,
    'commits_per_day': 10.0,
    'meetings_per_day': 5.0,
    'screen_time': 10.07,
    'exercise_hours': 1.00
}

df_new = pd.DataFrame([data])

df_new_scaled = scaler.transform(df_new)

pred = model.predict(df_new_scaled)
print(pred)

['Low']


Training with XGBoost